# 01 Quellen- und Clusterprüfung

## Zweck

Stellen Sie sicher, dass die drei erforderlichen Datenquellen und die verfügbare FH-Clusterumgebung machbar sind, bevor Sie mit der Implementierung fortfahren.

Bei diesem Notizbuch handelt es sich um ein Spike-Notizbuch. Die endgültige Pipeline, der Kafka-Produzent, der Spark-Streaming-Job und die Analyseschicht werden nicht implementiert.

## Eingaben

- Zwei Pilotstädte: Wien und Berlin.
- Open-Meteo Air Quality API-Endpunkt.
- Wikipedia-Stadtseiten.
- EEA historische Quellendokumentation zur Luftqualität und Erwartungen an lokale Dateien.
- FH Spark/Kafka-Umgebungsnotizen ohne Geheimnisse.

## Ausgaben

- Quellen-Machbarkeitsmatrix.
- Optionale winzige Open-Meteo JSON-Beispiele unter `data/bronze/open_meteo_raw/`.
- Optionale kleine Wikipedia-Beispiele HTML unter `data/bronze/wikipedia_html/`.
- Hinweise zur Cluster-Konnektivität und Speicherentscheidung.

## Verwendete Technologien

Python, Pandas, Requests, BeautifulSoup, JSON, Markdown und optionaler PySpark-Rauchtestcode.

## Konfiguration

Externe Quellaufrufe werden durch `RUN_SOURCE_SPIKES` geschützt. Legen Sie die Umgebungsvariable `RUN_SOURCE_SPIKES=true` fest, um die HTTP-Prüfungen auszuführen.

Der Spark-Cluster-Code wird nur als Smoke-Test-Vorlage bereitgestellt. Parquet-erzeugende Notebooks verwenden `SPARK_MASTER_URL=local[*]`, solange kein gemeinsamer Cluster-Speicher bestätigt ist.

In [ ]:
from pathlib import Path
import os
import json
import pandas as pd

_env_root = os.getenv("PROJECT_ROOT")
if _env_root:
    PROJECT_ROOT = Path(_env_root).resolve()
elif Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / Path(os.getenv("DATA_DIR", "data"))
CHECKPOINT_DIR = PROJECT_ROOT / Path(os.getenv("CHECKPOINT_DIR", "data/checkpoints"))

RUN_SOURCE_SPIKES = os.getenv("RUN_SOURCE_SPIKES", "false").lower() == "true"
OPEN_METEO_BASE_URL = "https://air-quality-api.open-meteo.com/v1/air-quality"
WIKIPEDIA_RAW_DIR = DATA_DIR / "bronze" / "wikipedia_html"
OPEN_METEO_RAW_DIR = DATA_DIR / "bronze" / "open_meteo_raw"
WIKIPEDIA_RAW_DIR.mkdir(parents=True, exist_ok=True)
OPEN_METEO_RAW_DIR.mkdir(parents=True, exist_ok=True)

pilot_cities = pd.DataFrame([
    {"city_id": "vienna_at", "city_name": "Vienna", "country_code": "AT", "latitude": 48.2082, "longitude": 16.3738, "wikipedia_url": "https://en.wikipedia.org/wiki/Vienna"},
    {"city_id": "berlin_de", "city_name": "Berlin", "country_code": "DE", "latitude": 52.5200, "longitude": 13.4050, "wikipedia_url": "https://en.wikipedia.org/wiki/Berlin"},
])
pilot_cities

## Implementierung

### Open-Meteo REST API Spitze

Die REST API-Prüfung fordert PM2.5, PM10 und NO2 unter Verwendung der Open-Meteo-Feldnamen `pm2_5`, `pm10` und `nitrogen_dioxide` an. Der Spike speichert nur winzige JSON-Beispieldateien, wenn die Ausführung aktiviert ist.

#### Definieren Sie die Open-Meteo-Anfrage

Dieser Helfer erstellt die API-Parameter und führt eine geschützte Anfrage für eine Stadt aus. Es speichert nur ein kleines Bronze-Beispiel und gibt einen strukturierten Statusdatensatz zurück, anstatt Anforderungsfehler auszublenden.

In [ ]:
def build_open_meteo_params(latitude: float, longitude: float) -> dict:
    return {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": "pm2_5,pm10,nitrogen_dioxide",
        "timezone": "UTC",
        "forecast_days": 1,
    }


def fetch_open_meteo_sample(city_row) -> dict:
    import requests

    try:
        response = requests.get(
            OPEN_METEO_BASE_URL,
            params=build_open_meteo_params(city_row["latitude"], city_row["longitude"]),
            timeout=20,
        )
        response.raise_for_status()
        payload = response.json()
        output_path = OPEN_METEO_RAW_DIR / f"{city_row['city_id']}_sample.json"
        output_path.write_text(__import__("json").dumps(payload, indent=2), encoding="utf-8")
        hourly_keys = sorted(payload.get("hourly", {}).keys())
        return {
            "source": "Open-Meteo",
            "city_id": city_row["city_id"],
            "status": "usable" if {"pm2_5", "pm10", "nitrogen_dioxide"}.issubset(hourly_keys) else "usable with constraints",
            "format": "JSON",
            "relevant_fields": ", ".join(hourly_keys),
            "evidence_path": str(output_path),
            "risks": "Feldverfügbarkeit und fehlende Stundenwerte müssen in Phase 5 geprüft werden.",
        }
    except requests.RequestException as exc:
        return {
            "source": "Open-Meteo",
            "city_id": city_row["city_id"],
            "status": f"Quellenprüfung fehlgeschlagen: {exc}",
            "format": "JSON",
            "relevant_fields": "pm2_5, pm10, nitrogen_dioxide",
            "evidence_path": "Abruf fehlgeschlagen",
            "risks": "Quelle muss erneut geprüft werden. Netzwerkzugriff und API-Verfügbarkeit prüfen.",
        }

print("OK: build_open_meteo_params, fetch_open_meteo_sample definiert")


#### Führen Sie die Open-Meteo-Pilotanfragen aus

Die Quellenspitze gilt nur für die beiden Pilotstädte. Externe Anrufe bleiben deaktiviert, es sei denn `RUN_SOURCE_SPIKES=true`; Dadurch bleibt ein normaler lokaler Run-All vorhersehbar.

In [ ]:
open_meteo_results = []
if RUN_SOURCE_SPIKES:
    for _, row in pilot_cities.iterrows():
        open_meteo_results.append(fetch_open_meteo_sample(row))
else:
    open_meteo_results.append({
        "source": "Open-Meteo",
        "city_id": "pilot",
        "status": "in diesem Lauf nicht ausgeführt",
        "format": "JSON",
        "relevant_fields": "pm2_5, pm10, nitrogen_dioxide",
        "evidence_path": "RUN_SOURCE_SPIKES=true setzen, um kleine Beispieldaten zu erzeugen",
        "risks": "Quelle muss bei der Ausführung von Phase 1 erneut geprüft werden.",
    })

pd.DataFrame(open_meteo_results)


### Wikipedia HTML Spitze

Der Web-Scraping-Spike prüft, ob die Stadt HTML mit einem eindeutigen User-Agent abgerufen und als roher Bronze-Beweis gespeichert werden kann.

#### Definieren Sie die Wikipedia-Pilotanfrage

Dieser Helfer lädt eine Stadtseite mit einem expliziten Benutzeragenten herunter, überprüft, ob HTML zurückgegeben wurde, speichert ein kleines Bronze-Artefakt und meldet Fehler als Daten.

In [ ]:
def fetch_wikipedia_sample(city_row) -> dict:
    import requests
    from bs4 import BeautifulSoup

    headers = {"User-Agent": "euro-air-quality-pipeline/1.0 didaktische Quellenprüfung"}
    try:
        response = requests.get(city_row["wikipedia_url"], headers=headers, timeout=20)
        response.raise_for_status()
        html = response.text
        output_path = WIKIPEDIA_RAW_DIR / f"{city_row['city_id']}_sample.html"
        output_path.write_text(html, encoding="utf-8")
        soup = BeautifulSoup(html, "html.parser")
        return {
            "source": "Wikipedia",
            "city_id": city_row["city_id"],
            "status": "usable" if soup.select_one("table.infobox") else "usable with constraints",
            "format": "HTML",
            "relevant_fields": "Infobox, Titel, mögliche Bevölkerungs-/Flächenfelder",
            "evidence_path": str(output_path),
            "risks": "Die HTML-Struktur kann sich ändern und Werte können mehrdeutig sein.",
        }
    except requests.RequestException as exc:
        return {
            "source": "Wikipedia",
            "city_id": city_row["city_id"],
            "status": f"Quellenprüfung fehlgeschlagen: {exc}",
            "format": "HTML",
            "relevant_fields": "infobox, population, area, coordinates",
            "evidence_path": "Abruf fehlgeschlagen",
            "risks": "Quelle muss erneut geprüft werden. Netzwerkzugriff prüfen.",
        }

print("OK: fetch_wikipedia_sample definiert")


#### Führen Sie die Wikipedia-Pilotanfragen aus

Die gleiche Pilot-Stadt-Schleife wird von `RUN_SOURCE_SPIKES` bewacht. Die zurückgegebenen Datensätze werden später Teil der Quellen-Machbarkeitsmatrix.

In [ ]:
wikipedia_results = []
if RUN_SOURCE_SPIKES:
    for _, row in pilot_cities.iterrows():
        wikipedia_results.append(fetch_wikipedia_sample(row))
else:
    wikipedia_results.append({
        "source": "Wikipedia",
        "city_id": "pilot",
        "status": "in diesem Lauf nicht ausgeführt",
        "format": "HTML",
        "relevant_fields": "infobox, population, area, coordinates",
        "evidence_path": "RUN_SOURCE_SPIKES=true setzen, um kleine Beispieldaten zu erzeugen",
        "risks": "Der Parser muss defensiv arbeiten und parse_status dokumentieren.",
    })

pd.DataFrame(wikipedia_results)


### EEA Datei-/Batch-Quellenprüfung

EEA ist die erforderliche Datei-/Stapelquelle. Das Projekt erwartet bei dieser Spitze einen lokalen CSV- oder Parquet-Extrakt, keinen automatischen vollständigen Download. Phase 3 implementiert Laden, Normalisierung und Aggregation.

In [ ]:
eea_source_check = pd.DataFrame([
    {
        "source": "Historische EEA-Luftqualität",
        "type": "file/batch",
        "status": "usable with constraints",
        "format": "Lokaler CSV- oder Parquet-Auszug",
        "relevant_fields": "Station, Zeitstempel, Schadstoff, Wert, Einheit",
        "risks": "Stations-zu-Stadt-Zuordnung, Schadstoffabdeckung, Dateiverfügbarkeit",
        "decision": "Als historische Datei-/Batch-Quelle in Notebook 03 verwenden.",
    }
])
eea_source_check


### FH Spark-Clusterprüfung

Bekanntes Ergebnis früherer Konnektivitätstests: Der Spark-Master war erreichbar und eine grundlegende DataFrame-Aktion funktionierte. HDFS/gemeinsamer Speicher wurde nicht bestätigt, `fs.defaultFS` wurde als `file:///` beobachtet und es wird nicht davon ausgegangen, dass lokale Jupyter-Pfade mit Executoren gemeinsam genutzt werden.

In [ ]:
cluster_findings = pd.DataFrame([
    {"check": "Spark-Master erreichbar", "status": "passed", "decision": "Cluster kann für Smoke-Tests verwendet werden"},
    {"check": "Grundlegende DataFrame-Aktion", "status": "passed", "decision": "Rechenverbindung ist dokumentiert"},
    {"check": "HDFS/gemeinsamer Speicher", "status": "nicht bestätigt", "decision": "Cluster nicht für Projektdaten oder Parquet-Ausgaben verwenden"},
    {"check": "Standardmodus der Pipeline", "status": "decided", "decision": "Spark local[*] für Notebooks mit Parquet-Ausgaben verwenden"},
])
cluster_findings


## Validierung / Qualitätsprüfungen

Überprüfen Sie, ob alle drei Datenquellen vertreten sind und dass die Cluster-Entscheidung den gemeinsam genutzten Speicher nicht überbeansprucht.

In [ ]:
source_matrix = pd.concat([
    pd.DataFrame(open_meteo_results),
    pd.DataFrame(wikipedia_results),
    eea_source_check.rename(columns={"type": "source_type"}),
], ignore_index=True, sort=False)

expected_sources = {"Open-Meteo", "Wikipedia", "Historische EEA-Luftqualität"}
missing_sources = expected_sources - set(source_matrix["source"])
assert not missing_sources, f"In der Quellenmatrix fehlen Einträge für: {missing_sources}"

hdfs_row = cluster_findings.loc[cluster_findings["check"].eq("HDFS/gemeinsamer Speicher"), "status"]
assert not hdfs_row.eq("passed").any(), \
    "HDFS/gemeinsamer Speicher darf nicht als bestanden markiert werden — gemeinsamer Clusterspeicher ist für dieses Projekt nicht bestätigt"

source_matrix

## Ergebnisse

In Phase 1 wird festgestellt, dass die geplanten Quellen realisierbar oder mit Einschränkungen realisierbar sind. Die tatsächlichen Quellausführungen können durch Aktivieren des Guard-Flags erneut ausgeführt werden, und generierte Beispiele bleiben von Git ignoriert.

## Einschränkungen

Die Quellenverfügbarkeit kann sich ändern. Die Datenverfügbarkeit von EEA hängt von lokalen Extrakten ab. Der FH-Cluster kann nicht für eine zuverlässige projektlokale Parquet-Ausgabe verwendet werden, es sei denn, der gemeinsame Speicher wird bestätigt.

## Nächster Schritt

Führen Sie das Notebook `02_city_reference_model.ipynb` aus, um das Central City Join-Modell zu erstellen.